___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://ichef.bbci.co.uk/ace/standard/3840/cpsprodpb/cea1/live/1de105b0-f5a5-11ef-bcea-7b70a14a5556.jpg" width="380px" height="180px" />


# <font color= #bbc28d> **Práctica FFNN - Momentum del Caos** </font>
#### <font color= #2E9AFE> `Tarea 5 - MNLP`</font>
- <Strong> Diana Valdivia, Samantha Sanchez, Clara Aguilar & Daniela de la Torre. </Strong>
- <Strong> Fecha </Strong>: 28/03/2026.

___

<p style="text-align:right;"> Image retrieved from: https://ichef.bbci.co.uk/ace/standard/3840/cpsprodpb/cea1/live/1de105b0-f5a5-11ef-bcea-7b70a14a5556.jpg</p>

In [1]:
# Importar librerías

import pandas as pd
import numpy as np
import requests
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# <font color=#bbc28d> **Introducción F1** </font>
Para esta tarea, nuestro propósito es intentar modelar y predecir el comportamiento dinámico de las carreras de Fórmula 1 mediante una métrica denominada **“Momentum de Caos”**. Esta métrica busca capturar el grado de variabilidad o interrupción en una carrera a partir del **número total de vueltas** completadas.



In [2]:
# Nuestra llave de la API
API_KEY = "a1a1af36c9e9508c4637511e46694c09"

# Endpoint de la F1
url = "https://v1.formula-1.api-sports.io/races"

headers = {'x-apisports-key': API_KEY}

# Tomar la info de los ultimos años
years = [2020, 2021, 2022, 2023, 2024, 2025]

# Filas de nuestr df
rows = []

for year in years:
  params = {"season": year,"type": "race"}

  response = requests.get(url, headers=headers, params=params)
  data = response.json()

  for race in data['response']:
    # Numero total de laps
    laps = race['laps']['total'] if race.get('laps') else None

    rows.append({'laps': laps,'date': race.get('date')})

df = pd.DataFrame(rows)
df

,laps,date
0,57,2022-03-20T15:00:00+00:00
1,58,2022-03-27T17:00:00+00:00
2,58,2022-04-10T05:00:00+00:00
3,63,2022-04-24T13:00:00+00:00
4,57,2022-05-08T19:30:00+00:00
...,...,...
66,71,2024-10-27T20:00:00+00:00
67,71,2024-11-03T15:30:00+00:00
68,50,2024-11-24T06:00:00+00:00
69,57,2024-12-01T16:00:00+00:00


Como podemos ver, no hay mucha información acerca de la F1, esto ya que es un evento que no suele ser tan cotidiano como otros deportes, sin embargo aún se puede trabajar con esta serie, pasaremos a realizar una limpieza básica de la serie:

In [3]:
# Limpieza básica
df = df[df['laps'].notna()]

# Convertir datos
df['laps'] = df['laps'].astype(float)
df['date'] = pd.to_datetime(df['date'])

# Ordenar la serie
df = df.sort_values('date')
df = df.reset_index(drop=True)
df

,laps,date
0,57.0,2022-03-20 15:00:00+00:00
1,58.0,2022-03-27 17:00:00+00:00
2,58.0,2022-04-10 05:00:00+00:00
3,63.0,2022-04-24 13:00:00+00:00
4,57.0,2022-05-08 19:30:00+00:00
...,...,...
66,71.0,2024-10-27 20:00:00+00:00
67,71.0,2024-11-03 15:30:00+00:00
68,50.0,2024-11-24 06:00:00+00:00
69,57.0,2024-12-01 16:00:00+00:00


La hipótesis es que las carreras con un número de vueltas **inferior** al valor típico `[mediana]` pueden estar asociadas a eventos como accidentes, banderas rojas o condiciones climáticas, que generan mayor incertidumbre en el desarrollo de la competencia.

In [4]:
# Crear la variable objetivo
df['finish'] = (df['laps'] < df['laps'].median()).astype(int)
df

# 1 = carrera terminó antes
# 0 = carrera con todas las vueltas

,laps,date,finish
0,57.0,2022-03-20 15:00:00+00:00,0
1,58.0,2022-03-27 17:00:00+00:00,0
2,58.0,2022-04-10 05:00:00+00:00,0
3,63.0,2022-04-24 13:00:00+00:00,0
4,57.0,2022-05-08 19:30:00+00:00,0
...,...,...,...
66,71.0,2024-10-27 20:00:00+00:00,0
67,71.0,2024-11-03 15:30:00+00:00,0
68,50.0,2024-11-24 06:00:00+00:00,1
69,57.0,2024-12-01 16:00:00+00:00,0


Según lo anterior construimos una variable binaria que identifica estas situaciones que generan "caos" y luego calculámos un promedio móvil para obtener una medida temporal del "caos" reciente.

Esta es nuestra target a predecir.

In [5]:
# Serie de tiempo de momentum
df_ts = df[['date', 'finish']].copy()

df_ts['Momentum'] = df_ts['finish'].rolling(window=5, min_periods=1).mean()
df_ts

,date,finish,Momentum
0,2022-03-20 15:00:00+00:00,0,0.0
1,2022-03-27 17:00:00+00:00,0,0.0
2,2022-04-10 05:00:00+00:00,0,0.0
3,2022-04-24 13:00:00+00:00,0,0.0
4,2022-05-08 19:30:00+00:00,0,0.0
...,...,...,...
66,2024-10-27 20:00:00+00:00,0,0.6
67,2024-11-03 15:30:00+00:00,0,0.4
68,2024-11-24 06:00:00+00:00,1,0.4
69,2024-12-01 16:00:00+00:00,0,0.4


Visulalizamos nuestra variable a predecir:

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_ts['date'],
    y=df_ts['Momentum'],
    name='Momentum de Caos',
    line=dict(color='#2ca02c', width=3)
))

fig.update_layout(
    title='Momentum de Caos en F1',
    xaxis_title='Fecha',
    yaxis_title='Promedio móvil de caos'
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [7]:
# Preparar la serie
serie_tiempo = df_ts['Momentum'].dropna().values.reshape(-1, 1)

# <font color=#bbc28d> **Modelado** </font>
Una vez creamos nuestra variable objetivo el siguiente paso es contruit nuestro modelo e intentar modelarla.

Para evitar el data leakage, procederemos a realizar nuestro train/test split antes de escalar y de crear nuestras ventanas para la red neuronal:

In [8]:
# Train test split
train_size = int(len(serie_tiempo) * 0.8)

train_data = serie_tiempo[:train_size]
test_data = serie_tiempo[train_size:]

Como vimos en clase, las series se benefician de escalar los valores entre ciertos rangos para que convergan más rápido por lo que nosotras también lo haremos:

In [9]:
# Escalamiento
scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

Para capturar la dependencia temporal de la serie, los datos se reorganizan en ventanas deslizantes, donde cada observación de entrada contiene los valores históricos recientes y la salida corresponde al siguiente valor temporal.

En nuestro caso probamos distintos valores, el mejor caso que nos dió fue de 5 por lo que nos quedaremos con ese:

In [10]:
# Crear las ventanas
def crear_ventanas(data, window_size):
    X, y = [], []

    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])

    return np.array(X), np.array(y)

# 5 días atras ( esto es un hiperparámetro)
window_size = 5

X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

Una vez lista la información, realizaremos nuestra red neuronal:

In [11]:
# Modelo FFNN
model = Sequential([
    Dense(32, activation='relu', input_dim=window_size),
    Dense(16, activation='relu'),
    Dense(1)])

model.summary()

model.compile(optimizer=Adam(learning_rate=0.001),loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 737 (2.88 KB)

 Trainable params: 737 (2.88 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Entrenar la red
history = model.fit(X_train, y_train, epochs=80, batch_size=16, validation_split=0.1, verbose=0)

In [ ]:
import matplotlib.pyplot as plt

# Graficar epoca vs pérdida
plt.figure()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

Una vez nuestra red FFNN esta entrenada, procederemos a realizar predicciones en nuestro test:

In [ ]:
# Predicción
y_pred_scaled = model.predict(X_test, verbose=0)

# Invertir escalamiento
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

# Métricas
mae = mean_absolute_error(y_test_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

Podemos ver que las métricas que dan son bastantes buenas, au que también hay que condierar que la escala de los datos es pequeña.

Visualicemos el forecast vs los valores reales del **Momentum del Caos**:

In [ ]:
fig = go.Figure()

# índices para alinear
train_index = range(window_size, train_size)
test_index = range(train_size + window_size, len(df_ts))

# TRAIN
fig.add_trace(go.Scatter(
    x=df_ts['date'].iloc[window_size:train_size],
    y=scaler.inverse_transform(y_train.reshape(-1,1)).flatten(),
    mode='lines',
    name='Entrenamiento (Real)',
    line=dict(color='#bdc3c7')
))

# TEST REAL
fig.add_trace(go.Scatter(
    x=df_ts['date'].iloc[train_size + window_size:],
    y=y_test_real.flatten(),
    mode='lines+markers',
    name='Real (Test)',
    line=dict(color='#2ca02c', width=2)
))

# PREDICCIÓN
fig.add_trace(go.Scatter(
    x=df_ts['date'].iloc[train_size + window_size:],
    y=y_pred.flatten(),
    mode='lines+markers',
    name='Predicción FFNN',
    line=dict(color='#e74c3c', dash='dot', width=2)
))

# línea divisoria
fig.add_vline(
    x=df_ts['date'].iloc[train_size],
    line_width=2,
    line_dash="dash",
    line_color="black"
)

fig.update_layout(
    title='Forecast FFNN - Momentum de Caos en F1',
    xaxis_title='Fecha',
    yaxis_title='Momentum de Caos'
)

fig.show()